In [29]:
%pip install -U langchain_community langchain_groq

In [30]:
import os
from google.colab import userdata
api_key = userdata.get("groq_api_key")
os.environ["GROQ_API_KEY"] = api_key

**import Libraries**

In [31]:
import re
import json
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage
from typing import Callable, Dict, Any, Optional

**ENHANCED FUNCTION CALLING SCHEMA**

In [32]:
DEFAULT_FUNCTION_SCHEMA = {
    "name": "extract_collaboration_workflow",
    "description": "Extract structured workflow info from a natural language collaboration prompt",
    "parameters": {
        "type": "object",
        "properties": {
            "phases": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "pattern": {
                            "type": "string",
                            "enum": ["parallel", "sequential", "conditional", "loop"]
                        },
                        "agents": {"type": "array", "items": {"type": "string"}},
                        "duration": {
                            "type": "string",
                            "enum": ["until_complete", "fixed_time", "conditional", "ongoing"]
                        },
                        "conditions": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["pattern", "agents", "duration"]
                }
            },
            "rules": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "type": {
                            "type": "string",
                            "enum": ["conditional_loop", "dependency", "validation", "retry"]
                        },
                        "condition": {"type": "string"},
                        "action": {"type": "string"}
                    },
                    "required": ["type", "condition", "action"]
                }
            },
            "dependencies": {
                "type": "object",
                "additionalProperties": {"type": "array", "items": {"type": "string"}}
            },
            "roles": {
                "type": "object",
                "additionalProperties": {"type": "string"}
            }
        },
        "required": ["phases", "rules", "dependencies", "roles"]
    }
}

def build_enhanced_function_call_prompt(prompt: str):
    system_content = f"""You are an expert workflow parser for multi-agent collaboration systems.

Your task is to extract structured workflow information from natural language prompts.

IMPORTANT GUIDELINES:
1. **Phases**: Identify collaboration patterns (parallel, sequential, conditional, loop)
2. **Duration**: Use meaningful durations like "until_complete", not generic times
3. **Rules**: Extract conditional logic, feedback loops, and validation rules
4. **Dependencies**: Map which agents depend on others' completion
5. **Roles**: Infer agent responsibilities from context

PATTERN EXAMPLES:
- "A and B work in parallel" → pattern: "parallel"
- "then C reviews" → pattern: "sequential"
- "if issues found, return to relevant agent" → rule: conditional_loop
- "QA reviews both outputs" → dependencies: QA depends on previous agents

SCHEMA TO FOLLOW:
{json.dumps(DEFAULT_FUNCTION_SCHEMA['parameters'], indent=2)}

EXAMPLE INPUT: "UI and Backend agents work in parallel, then QA agent reviews both outputs. If issues found, relevant agent fixes them."

EXAMPLE OUTPUT:
{{
  "phases": [
    {{
      "pattern": "parallel",
      "agents": ["UI", "Backend"],
      "duration": "until_complete",
      "conditions": []
    }},
    {{
      "pattern": "sequential",
      "agents": ["QA"],
      "duration": "until_complete",
      "conditions": ["all_previous_complete"]
    }}
  ],
  "rules": [
    {{
      "type": "conditional_loop",
      "condition": "issues_found",
      "action": "return_to_relevant_agent"
    }}
  ],
  "dependencies": {{
    "QA": ["UI", "Backend"]
  }},
  "roles": {{
    "UI": "Interface development",
    "Backend": "API and data processing",
    "QA": "Quality validation and testing"
  }}
}}

Return ONLY valid JSON matching the schema. No explanations."""

    return [
        SystemMessage(content=system_content),
        HumanMessage(content=f"Parse this collaboration prompt: {prompt}")
    ]

**ENHANCED HEURISTIC PARSER**

In [33]:
def heuristic_parse(prompt: str) -> Dict[str, Any]:
    """Enhanced fallback parser with better pattern detection"""

    # Common agent role mappings
    ROLE_MAPPINGS = {
        'ui': 'Interface development and user experience',
        'frontend': 'Frontend development and user interface',
        'backend': 'API development and data processing',
        'qa': 'Quality validation and testing',
        'testing': 'Quality assurance and validation',
        'review': 'Code review and quality control',
        'database': 'Data storage and management',
        'api': 'API development and integration',
        'security': 'Security validation and compliance'
    }

    # Extract agents
    agents = re.findall(r'\b([A-Z][A-Za-z]*)\s+(?:agent|team|developer)', prompt, re.IGNORECASE)
    if not agents:
        agents = re.findall(r'\b(frontend|backend|UI|QA|testing|database|api|security)\b', prompt, re.IGNORECASE)

    agents = [agent.title() for agent in agents]

    # Detect patterns and build phases
    phases = []
    dependencies = {}
    rules = []

    # Pattern detection
    if re.search(r'\b(parallel|simultaneously|at the same time)\b', prompt, re.IGNORECASE):
        parallel_agents = []
        for agent in agents:
            if agent.lower() in prompt.lower():
                parallel_agents.append(agent)

        if len(parallel_agents) >= 2:
            phases.append({
                "pattern": "parallel",
                "agents": parallel_agents[:2],  # First two agents work in parallel
                "duration": "until_complete",
                "conditions": []
            })

    # Sequential detection
    if re.search(r'\b(then|after|once|following)\b', prompt, re.IGNORECASE):
        remaining_agents = [a for a in agents if a not in (phases[0]["agents"] if phases else [])]
        if remaining_agents:
            sequential_agent = remaining_agents[0]
            phases.append({
                "pattern": "sequential",
                "agents": [sequential_agent],
                "duration": "until_complete",
                "conditions": ["all_previous_complete"]
            })

            # Set up dependencies
            if phases:
                dependencies[sequential_agent] = phases[0]["agents"] if phases[0]["pattern"] == "parallel" else []

    # Conditional rules detection
    if re.search(r'\b(if.*issues?.*found|if.*problems?|when.*errors?)\b', prompt, re.IGNORECASE):
        rules.append({
            "type": "conditional_loop",
            "condition": "issues_found",
            "action": "return_to_relevant_agent"
        })

    # Generate roles
    roles = {}
    for agent in agents:
        agent_lower = agent.lower()
        if agent_lower in ROLE_MAPPINGS:
            roles[agent] = ROLE_MAPPINGS[agent_lower]
        else:
            roles[agent] = f"{agent} development and implementation"

    return {
        "phases": phases,
        "rules": rules,
        "dependencies": dependencies,
        "roles": roles
    }

**LLM CALLER USING LANGCHAIN + GROQ**

In [39]:
def groq_langchain_caller(messages):
    llm = ChatGroq(
        temperature=0.1,
        model_name="llama3-8b-8192",
        verbose=True
    )

    try:
        response = llm.invoke(messages)

        # Clean the response content
        content = response.content.strip()

        # Remove markdown code blocks if present
        if content.startswith("```json"):
            content = content[7:]
        if content.endswith("```"):
            content = content[:-3]
        content = content.strip()

        result_json = json.loads(content)
        return {"arguments": result_json}

    except json.JSONDecodeError as e:
        print(f"[WARN] Failed to parse LLM JSON: {e}")
        print(f"[DEBUG] Raw response: {response.content}")
        return None
    except Exception as e:
        print(f"[ERROR] LLM call failed: {e}")
        return None

**MAIN PARSER**

In [35]:
def parse_collaboration_prompt(prompt: str, llm_caller: Optional[Callable] = None) -> Dict[str, Any]:
    """
    Parse natural language collaboration prompt into structured workflow specification.

    Args:
        prompt: Natural language description of collaboration workflow
        llm_caller: Optional LLM function for enhanced parsing

    Returns:
        Dictionary containing phases, rules, dependencies, and roles
    """

    if llm_caller:
        try:
            messages = build_enhanced_function_call_prompt(prompt)
            result = llm_caller(messages)
            if result and "arguments" in result:
                # Validate the result has required fields
                args = result["arguments"]
                if all(key in args for key in ["phases", "rules", "dependencies", "roles"]):
                    return args
                else:
                    print("[WARN] LLM result missing required fields, falling back to heuristic")
        except Exception as e:
            print(f"[WARN] LLM parsing failed: {e}")

    # Fallback to heuristic parser
    print("[INFO] Using heuristic parser")
    return heuristic_parse(prompt)

**VALIDATION FUNCTIONS**

In [36]:
def validate_workflow_spec(spec: Dict[str, Any]) -> bool:
    """Validate that the workflow specification is well-formed"""
    required_keys = ["phases", "rules", "dependencies", "roles"]

    if not all(key in spec for key in required_keys):
        return False

    # Validate phases
    for phase in spec["phases"]:
        if not all(key in phase for key in ["pattern", "agents", "duration"]):
            return False
        if phase["pattern"] not in ["parallel", "sequential", "conditional", "loop"]:
            return False

    return True

**SELF-TEST**

In [38]:
if __name__ == "__main__":
    # Test cases
    test_prompts = [
        "UI and Backend agents work in parallel, then QA agent reviews both outputs. If issues found, relevant agent fixes them.",
        "first frontend and QA work in parallel. then backend and QA in parallel.",
        "Frontend team develops the interface while Backend team creates APIs. Once both are done, QA team tests everything. If bugs are found, the respective team fixes them."
    ]

    for i, test_prompt in enumerate(test_prompts, 1):
        print(f"\n=== TEST {i} ===")
        print(f"Input: {test_prompt}")

        # Test with LangChain + Groq
        try:
            spec_llm = parse_collaboration_prompt(test_prompt, llm_caller=groq_langchain_caller)
            print("\nLLM output:")
            print(json.dumps(spec_llm, indent=2))

            # Validate
            if validate_workflow_spec(spec_llm):
                print("✅ Validation passed")
            else:
                print("❌ Validation failed")

        except Exception as e:
            print(f"❌ Test failed: {e}")

        print("-" * 50)


=== TEST 1 ===
Input: UI and Backend agents work in parallel, then QA agent reviews both outputs. If issues found, relevant agent fixes them.

LLM output:
{
  "phases": [
    {
      "pattern": "parallel",
      "agents": [
        "UI",
        "Backend"
      ],
      "duration": "until_complete",
      "conditions": []
    },
    {
      "pattern": "sequential",
      "agents": [
        "QA"
      ],
      "duration": "until_complete",
      "conditions": [
        "all_previous_complete"
      ]
    }
  ],
  "rules": [
    {
      "type": "conditional_loop",
      "condition": "issues_found",
      "action": "return_to_relevant_agent"
    }
  ],
  "dependencies": {
    "QA": [
      "UI",
      "Backend"
    ]
  },
  "roles": {
    "UI": "Interface development",
    "Backend": "API and data processing",
    "QA": "Quality validation and testing"
  }
}
✅ Validation passed
--------------------------------------------------

=== TEST 2 ===
Input: first frontend and QA work in parall